# Models — does review text help predict price?

Compares XGBoost price models that share an identical structured baseline
and differ only in which text-derived feature block they add. All feature
tables are precomputed and loaded here — no encoding/scoring/PCA happens in
this notebook:

- `features_basic.parquet`          (from `etl/04_feature_engineering.ipynb`)
- `features_keywords.parquet`       (from `etl/05_nlp_keywords.ipynb`)
- `features_embeddings_PCA.parquet` (from `etl/06_nlp_embeddings.ipynb` → `etl/07_PCA_reduction.ipynb`)

joined on `wine_id`:

- **Model 1** — base structured features only (the baseline to beat)
- **Model 2** — base + 8 keyword aroma densities
- **Model 3** — base + 20 embedding principal components

In [9]:
import numpy as np
import pandas as pd
import itables
from itables import show
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

itables.options.columnDefs = [{"className": "dt-left", "targets": "_all"}]

FEATURES_BASIC_PATH = r"..\..\.data\features_basic.parquet"
KEYWORDS_PATH = r"..\..\.data\features_keywords.parquet"
EMBEDDINGS_PCA_PATH = r"..\..\.data\features_embeddings_PCA.parquet"


## Load features

In [10]:
features = pd.read_parquet(FEATURES_BASIC_PATH)
keywords = pd.read_parquet(KEYWORDS_PATH)
embeddings_pca = pd.read_parquet(EMBEDDINGS_PCA_PATH)
print(f"features: {features.shape}  |  keywords: {keywords.shape}  |  embeddings_pca: {embeddings_pca.shape}")

target = "retail"

base_features = [
    "rating", "alcohol", "bottle_size", "vintage", "case_production",
    "country_ord", "wine_type_ord", "state_ord", "company_ord",
    "appellation_ord", "varietal_label_ord", "age_at_review",
]
base_features_excl_rating = [f for f in base_features if f != "rating"]

kw_features  = [c for c in keywords.columns if c.startswith("kw_") and not c.endswith("_count")]
pca_features = [c for c in embeddings_pca.columns if c.startswith("pca_")]

df = (
    features
    .merge(keywords[["wine_id"] + kw_features], on="wine_id", how="left")
    .merge(embeddings_pca[["wine_id"] + pca_features], on="wine_id", how="left")
)
assert len(df) == len(features), "merge changed row count — wine_id not unique?"
print(f"merged: {df.shape}")
print(f"basic features ({len(base_features)}): {base_features}")
print(f"aroma features ({len(kw_features)}): {kw_features}")
print(f"emb-PCA features ({len(pca_features)}): {pca_features}")


features: (135192, 15)  |  keywords: (135192, 17)  |  embeddings_pca: (135192, 21)
merged: (135192, 43)
basic features (12): ['rating', 'alcohol', 'bottle_size', 'vintage', 'case_production', 'country_ord', 'wine_type_ord', 'state_ord', 'company_ord', 'appellation_ord', 'varietal_label_ord', 'age_at_review']
aroma features (8): ['kw_fruity', 'kw_tannic', 'kw_acidic', 'kw_oaky', 'kw_sweet', 'kw_body', 'kw_earthy', 'kw_floral']
emb-PCA features (20): ['pca_00', 'pca_01', 'pca_02', 'pca_03', 'pca_04', 'pca_05', 'pca_06', 'pca_07', 'pca_08', 'pca_09', 'pca_10', 'pca_11', 'pca_12', 'pca_13', 'pca_14', 'pca_15', 'pca_16', 'pca_17', 'pca_18', 'pca_19']


## Train & compare

All models train on the *same* rows and split (`random_state=42`) so any
metric difference is attributable to the added feature block:

- **model 1** — `base` (structured features incl. `age_at_review`)
- **model 2** — `base + kw` (heuristic aroma keyword densities)
- **model 3** — `base + emb-PCA` (20 PCs of the review sentence embeddings)

In [11]:
# one big model_df with all feature blocks; same rows for every model
model_df = df[base_features + kw_features + pca_features + [target]].dropna(subset=[target])
low, high = model_df[target].quantile([0.02, 0.90])
model_df = model_df[model_df[target].between(low, high)]
print(f"Retail range after trimming: ${low:.2f} - ${high:.2f}  ({len(model_df):,} rows)")

# function to train and evaluate a model given a list of features
def train_eval(feature_list):
    X, y = model_df[feature_list], model_df[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = xgb.XGBRegressor(random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    return model, {
        "n_features": len(feature_list),
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "MAE":  mean_absolute_error(y_test, pred),
        "R2":   r2_score(y_test, pred),
    }

Retail range after trimming: $11.00 - $80.00  (113,220 rows)


### With `rating` (full base)

In [12]:
# train and evaluate models with different feature sets, including rating
model1, m1 = train_eval(base_features)
model2, m2 = train_eval(base_features + kw_features)
model3, m3 = train_eval(base_features + pca_features)

results = pd.DataFrame([
    {"model": "1: baseline",            **m1},
    {"model": "2: baseline + kw",       **m2},
    {"model": "3: baseline + emb-PCA",  **m3},
])
results["dR2_vs_base"] = results["R2"] - m1["R2"]
results.round(4)

,model,n_features,RMSE,MAE,R2,dR2_vs_base
0,1: baseline,12,10.4712,7.7238,0.6527,0.0000
1,2: baseline + kw,20,10.6005,7.8440,0.6440,-0.0086
2,3: baseline + emb-PCA,32,10.7885,8.0057,0.6313,-0.0214


### Without `rating`

Drops `rating` from `base` and retrains all three models. Compares how much
price prediction depends on the reviewer's score, and whether the text blocks
(`kw` / `emb-PCA`) recover signal in its absence. `dR2_vs_base` here is relative
to the no-rating baseline (model 4).

In [13]:
# Same three models, but with `rating` dropped from the feature set.
# rating is the strongest structured predictor of price, so this shows how much
# price prediction leans on the score — and whether the text blocks recover any
# of the lost signal when rating is unavailable.
model4, m4 = train_eval(base_features_excl_rating)
model5, m5 = train_eval(base_features_excl_rating + kw_features)
model6, m6 = train_eval(base_features_excl_rating + pca_features)

results_excl = pd.DataFrame([
    {"model": "4: no-rating base",       **m4},
    {"model": "5: no-rating + kw",       **m5},
    {"model": "6: no-rating + emb-PCA",  **m6},
])
results_excl["dR2_vs_base"] = results_excl["R2"] - m4["R2"]
results_excl.round(4)

,model,n_features,RMSE,MAE,R2,dR2_vs_base
0,4: no-rating base,11,11.2638,8.3669,0.5981,0.0000
1,5: no-rating + kw,19,11.3874,8.5015,0.5892,-0.0089
2,6: no-rating + emb-PCA,31,11.7273,8.7903,0.5643,-0.0338


## Feature importance — model 3

Where the 20 embedding PCs land relative to the structured features, and
how much total gain the `emb-PCA` block captures vs `base`.

In [14]:
imp = (
    pd.DataFrame({"feature": base_features + pca_features, "importance": model3.feature_importances_})
    .assign(group=lambda d: np.where(d["feature"].str.startswith("pca_"), "emb-PCA", "base"))
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
    .assign(importance=lambda d: d["importance"].round(3))
)

print("total importance by block:")
print(imp.groupby("group")["importance"].sum().round(3).to_string())
show(imp)

total importance by block:
group
base       0.838
emb-PCA    0.160


Loading ITables v2.7.3 from the internet... (need help?)


## Conclusion

- **Model 2 (keywords)** did not help — the 8 `kw_*` densities encode wine
  *style* already captured by `wine_type`/`varietal_label`, and as weak noisy
  inputs they let an untuned XGBoost overfit slightly (test R² down ~0.01).
- **Model 3 (embedding PCs)** is the real test of whether the review text
  carries price signal beyond the structured features. Read the table above:
  if `dR2_vs_base` is positive, the semantic content helps; the importance
  block-sum shows how much the model leans on `emb-PCA` vs `base`.

Follow-ups if model 3 helps: tune `N_COMPONENTS` in `07_PCA_reduction.ipynb`
(32/50), add light regularisation, then move to MLflow-tracked CV (Sprint 7).
If it doesn't: the review prose adds little once rating/varietal/region are
known — a finding worth reporting in itself.